# Урок 3.4 — JOIN в Spark: как соединять таблицы безопасно (DataFrame API + Spark SQL)

Этот ноутбук — урок 3.4 модуля «Основы Spark» для стенда `spark_01`.

В этом уроке вы впервые осознанно потрогаете JOIN **как инженер**:

- поймёте, чем отличаются `inner / left / right / full / anti / semi`;
- увидите, как JOIN “взорвает” строки при неуникальных ключах;
- научитесь проверять, что JOIN получился “правильным” (row count, уникальность, missing refs);
- сделаете то же самое через **Spark SQL**.

---

## С какими файлами работаем (внутри контейнера)

- `/data/csv/olist_orders_dataset.csv` — заказы
- `/data/csv/olist_order_items_dataset.csv` — позиции заказа
- `/data/csv/olist_customers_dataset.csv` — клиенты
- `/data/csv/olist_products_dataset.csv` — товары

---

## Правило путей в стенде

- исходные данные читаем из `/data/csv`
- результаты пишем в `/workspace/lesson03_03/...`

Причина: и driver в Jupyter, и executors на воркерах должны видеть одинаковые пути.


---
## 0. Импорты и SparkSession

В стенде `spark_01` Spark-сессия обычно уже доступна как `spark`.
Если переменной нет — создадим её.

💡 Любую ячейку можно перезапускать — DataFrame просто пересоздастся/переприсвоится.


In [ ]:
# Импорты, которые нам понадобятся в уроке
import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# Если spark уже есть — используем его. Если нет — создаём.
try:
    spark  # noqa: F821
except NameError:
    spark = SparkSession.builder.getOrCreate()

spark


---
## 1. Пути и хелперы

Spark пишет в **директорию**, внутри будут `part-...` файлы — это нормально.

Мы заведём 3 простые функции:
- `reset_dir(path)` — очистить папку перед записью
- `read_csv_dir(path)` — прочитать CSV из директории
- `read_parquet_dir(path)` — прочитать Parquet из директории


In [ ]:
# Пути в стенде
DATA_DIR = "/data/csv"
WORKSPACE_DIR = "/workspace"
LESSON_DIR = f"{WORKSPACE_DIR}/lesson03_03"

os.makedirs(LESSON_DIR, exist_ok=True)

# Источники
ORDERS_CSV = f"{DATA_DIR}/olist_orders_dataset.csv"
ORDER_ITEMS_CSV = f"{DATA_DIR}/olist_order_items_dataset.csv"
CUSTOMERS_CSV = f"{DATA_DIR}/olist_customers_dataset.csv"
PRODUCTS_CSV = f"{DATA_DIR}/olist_products_dataset.csv"

def reset_dir(path: str) -> None:
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)

def read_csv_dir(path: str):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

def read_parquet_dir(path: str):
    return spark.read.parquet(path)

(LESSON_DIR, ORDERS_CSV, ORDER_ITEMS_CSV, CUSTOMERS_CSV, PRODUCTS_CSV)


---
## 2. Мини-пример: что такое JOIN и почему он может “взорвать” строки

Сначала потрогаем JOIN на игрушечных DataFrame (5–6 строк).
Это самый быстрый способ понять:

- почему `inner` может “выкинуть” строки;
- почему `left` сохраняет левую таблицу;
- почему неуникальные ключи дают **мультипликацию** строк (many-to-many).

Дальше те же идеи перенесём на Olist.


In [ ]:
# Соберём два маленьких DataFrame
# (заметьте: ключ 'k' НЕ уникальный — это сделано специально)

df_left = spark.createDataFrame(
    [
        ("A", 1),
        ("A", 2),
        ("B", 10),
        ("C", 100),
    ],
    ["k", "v_left"]
)

df_right = spark.createDataFrame(
    [
        ("A", "x"),
        ("A", "y"),
        ("B", "z"),
        ("D", "w"),
    ],
    ["k", "v_right"]
)

df_left.show(truncate=False)
df_right.show(truncate=False)


In [ ]:
# INNER JOIN: остаются только ключи, которые есть И слева, И справа
df_inner = df_left.join(df_right, on="k", how="inner")

# Важно: для k='A' слева 2 строки и справа 2 строки → итог 2*2 = 4 строки
df_inner.orderBy("k", "v_left", "v_right").show(truncate=False)
df_inner.count()


In [ ]:
# LEFT JOIN: сохраняем ВСЕ строки слева, даже если справа нет совпадений
df_left_join = df_left.join(df_right, on="k", how="left")

df_left_join.orderBy("k", "v_left", "v_right").show(truncate=False)
(df_left.count(), df_left_join.count())


In [ ]:
# LEFT ANTI: строки слева, для которых НЕТ совпадения справа
df_left_anti = df_left.join(df_right, on="k", how="left_anti")

df_left_anti.orderBy("k", "v_left").show(truncate=False)


In [ ]:
# LEFT SEMI: строки слева, для которых ЕСТЬ совпадение справа
# (это как фильтр "оставь только те, кто есть в правой таблице", без колонок справа)

df_left_semi = df_left.join(df_right, on="k", how="left_semi")
df_left_semi.orderBy("k", "v_left").show(truncate=False)


---
## 3. Читаем таблицы Olist

Дальше работаем с реальными таблицами и реальными ключами.

Считываем:
- orders
- order_items
- customers
- products

Для обучения используем `inferSchema=True`.


In [ ]:
# Читаем заказы
df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

# Читаем позиции заказов
df_order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDER_ITEMS_CSV)
)

# Читаем клиентов
df_customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMERS_CSV)
)

# Читаем товары
df_products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PRODUCTS_CSV)
)

(df_orders, df_order_items, df_customers, df_products)


---
## 4. Inspect: минимальные проверки перед JOIN

Перед любыми JOIN полезно быстро ответить на 3 вопроса:

1) сколько строк в каждой таблице  
2) какие колонки есть (особенно ключи)  
3) уникален ли ключ, по которому вы хотите join-ить


In [ ]:
# Размеры таблиц
orders_cnt = df_orders.count()
items_cnt = df_order_items.count()
customers_cnt = df_customers.count()
products_cnt = df_products.count()

orders_cnt, items_cnt, customers_cnt, products_cnt


In [ ]:
# Посмотрим колонки-ключи (по 5 строк)
df_orders.select("order_id", "customer_id", "order_status", "order_purchase_timestamp").show(5, truncate=False)
df_order_items.select("order_id", "order_item_id", "product_id", "price").show(5, truncate=False)
df_customers.select("customer_id", "customer_unique_id", "customer_state").show(5, truncate=False)
df_products.select("product_id", "product_category_name").show(5, truncate=False)


---
## 5. Инженерная проверка перед JOIN: уникальность ключей

Если ключ НЕ уникальный с обеих сторон — возможен many-to-many и “взрыв” строк.

В Olist обычно:
- `orders.order_id` уникален в `orders`
- `order_items.order_id` НЕ уникален (у заказа несколько позиций)
- `customers.customer_id` уникален в `customers`
- `products.product_id` уникален в `products`

Проверим это явно.


In [ ]:
# Проверка уникальности ключей (простая и наглядная)
orders_order_id_distinct = df_orders.select("order_id").distinct().count()
items_order_id_distinct = df_order_items.select("order_id").distinct().count()

customers_customer_id_distinct = df_customers.select("customer_id").distinct().count()
products_product_id_distinct = df_products.select("product_id").distinct().count()

(orders_cnt, orders_order_id_distinct), (items_cnt, items_order_id_distinct), (customers_cnt, customers_customer_id_distinct), (products_cnt, products_product_id_distinct)


---
## 6. JOIN “заказы + позиции”: самый частый кейс

Это классика аналитики:

- `orders` — “шапка”
- `order_items` — “строки/позиции”

Тип JOIN почти всегда `inner`.

Сначала сделаем **узкий join** (выбираем только нужные колонки).


In [ ]:
# Узкие таблицы (контракты) для JOIN
df_orders_narrow = df_orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
)

df_items_narrow = df_order_items.select(
    "order_id",
    "order_item_id",
    "product_id",
    "price"
)

# Узкий inner join: items + orders по order_id
df_orders_items = df_items_narrow.join(df_orders_narrow, on="order_id", how="inner")

df_orders_items.show(5, truncate=False)
(df_items_narrow.count(), df_orders_items.count())


In [ ]:
# Проверка: есть ли позиции, у которых НЕТ "шапки" заказа? (left_anti)
df_items_without_orders = (
    df_items_narrow
    .join(df_orders_narrow.select("order_id"), on="order_id", how="left_anti")
)

df_items_without_orders.count()


---
## 7. JOIN “заказы + клиенты”: enrichment

Когда хотим добавить признаки клиента к заказу, обычно делаем `left`:
- сохраняем все заказы
- если клиент не найдётся, увидим NULL и сможем разобрать проблему

Важно: решаем коллизии имён через `select`.


In [ ]:
# Узкий справочник клиентов
df_customers_narrow = df_customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
)

# LEFT JOIN: orders -> customers
df_orders_customers = df_orders_narrow.join(df_customers_narrow, on="customer_id", how="left")

df_orders_customers.show(5, truncate=False)

# Сколько заказов "потеряли" клиента (NULL customer_unique_id)?
df_orders_customers.filter(F.col("customer_unique_id").isNull()).count()


---
## 8. Цепочка JOIN: orders + items + products (обогащаем категориями)

Теперь усложняем:

1) соединяем позиции с заказами  
2) добавляем к каждой позиции категорию товара (`products`)


In [ ]:
# Узкий справочник товаров
df_products_narrow = df_products.select(
    "product_id",
    "product_category_name"
)

# LEFT join к products: позиции важнее справочника
df_items_enriched = (
    df_orders_items
    .join(df_products_narrow, on="product_id", how="left")
)

df_items_enriched.select(
    "order_id", "order_item_id", "product_id", "product_category_name", "price", "order_status"
).show(5, truncate=False)

# Сколько позиций без категории (NULL category)?
df_items_enriched.filter(F.col("product_category_name").isNull()).count()


---
## 9. То же самое через Spark SQL

1) регистрируем DataFrame как temp view  
2) пишем SQL  
3) сравниваем результаты с DataFrame API (хотя бы по `count()`)

Зарегистрируем view:
- `orders_narrow`
- `items_narrow`
- `products_narrow`


In [ ]:
# Регистрируем temp views для SQL
df_orders_narrow.createOrReplaceTempView("orders_narrow")
df_items_narrow.createOrReplaceTempView("items_narrow")
df_products_narrow.createOrReplaceTempView("products_narrow")


In [ ]:
# SQL: orders + items + products (аналог df_items_enriched)
query_items_enriched = '''
SELECT
    i.order_id,
    i.order_item_id,
    i.product_id,
    p.product_category_name,
    i.price,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM items_narrow i
JOIN orders_narrow o
    ON i.order_id = o.order_id
LEFT JOIN products_narrow p
    ON i.product_id = p.product_id
'''

df_items_enriched_sql = spark.sql(query_items_enriched)

df_items_enriched_sql.show(5, truncate=False)
(df_items_enriched.count(), df_items_enriched_sql.count())


---
## 10. Идемпотентная запись примера + read-back

Запишем `df_items_enriched` в Parquet и прочитаем обратно.


In [ ]:
# Путь для примера (не практика)
OUT_ITEMS_ENRICHED_PARQUET = f"{LESSON_DIR}/items_enriched_parquet"

# Пишем идемпотентно: чистим директорию и пишем overwrite
reset_dir(OUT_ITEMS_ENRICHED_PARQUET)

(
    df_items_enriched
    .write
    .mode("overwrite")
    .parquet(OUT_ITEMS_ENRICHED_PARQUET)
)

# read-back
df_items_enriched_back = read_parquet_dir(OUT_ITEMS_ENRICHED_PARQUET)

df_items_enriched.count(), df_items_enriched_back.count()


---
# Практика

Ниже — 8 заданий про JOIN.

Правило для каждого задания:
1) трансформация (JOIN/фильтр/агрегация)
2) `show(5, truncate=False)`
3) запись в указанный путь
4) read-back + проверки

Важно:
- порядок строк после записи/чтения **не гарантирован**
- для проверок мы сортируем там, где нужно


In [ ]:
# Пути для практики (используются и в проверках)
TASK01_OUT = f"{LESSON_DIR}/task01_orders_items_inner_parquet"
TASK02_OUT = f"{LESSON_DIR}/task02_items_products_left_parquet"
TASK03_OUT = f"{LESSON_DIR}/task03_items_without_orders_csv"
TASK04_OUT = f"{LESSON_DIR}/task04_orders_with_customers_left_parquet"
TASK05_OUT = f"{LESSON_DIR}/task05_orders_items_cnt_parquet"
TASK06_OUT = f"{LESSON_DIR}/task06_chain_orders_items_products_parquet"
TASK07_OUT = f"{LESSON_DIR}/task07_top_categories_parquet"
TASK08_OUT = f"{LESSON_DIR}/task08_debug_sample_parquet"

(TASK01_OUT, TASK02_OUT, TASK03_OUT)


## Задание 1. INNER JOIN orders + order_items

Вход:
- `df_orders_narrow`
- `df_items_narrow`

Что сделать:
- inner join по `order_id`
- вывести: `order_id`, `order_item_id`, `product_id`, `price`, `customer_id`, `order_status`

Куда записать:
- Parquet: `/workspace/lesson03_03/task01_orders_items_inner_parquet`


In [ ]:
df_orders_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

df_items_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDER_ITEMS_CSV)
)

print(f"df_orders_narrow: {df_orders_narrow.count()}, df_items_narrow: {df_items_narrow.count()}")

df_join = df_orders_narrow.join(df_items_narrow, on="order_id", how="inner")\
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "price",
        "customer_id",
        "order_status"
    )

df_join.show(5, truncate=False)

reset_dir(TASK01_OUT)

(
    df_join
    .write
    .mode("overwrite")
    .parquet(TASK01_OUT)
)

df_join_back = read_parquet_dir(TASK01_OUT)

print(f"df_join: {df_join.count()}, df_join_back: {df_join_back.count()}")

## Задание 2. LEFT JOIN order_items + products

Вход:
- `df_items_narrow`
- `df_products_narrow`

Что сделать:
- left join по `product_id`
- вывести: `order_id`, `order_item_id`, `product_id`, `product_category_name`, `price`

Куда записать:
- Parquet: `/workspace/lesson03_03/task02_items_products_left_parquet`


In [ ]:
df_products_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PRODUCTS_CSV)
)

df_order_items_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDER_ITEMS_CSV)
)

print(f"df_products_narrow: {df_products_narrow.count()}, df_order_items_narrow: {df_order_items_narrow.count()}")

df_join = df_products_narrow.join(df_order_items_narrow, on="product_id", how="left")\
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "product_category_name",
        "price"
    )

df_join.show(5, truncate=False)

reset_dir(TASK02_OUT)

(
    df_join
    .write
    .mode("overwrite")
    .parquet(TASK02_OUT)
)

df_join_back = read_parquet_dir(TASK02_OUT)

print(f"df_join: {df_join.count()}, df_join_back: {df_join_back.count()}")

## Задание 3. LEFT ANTI: позиции без заказа

Вход:
- `df_items_narrow`
- `df_orders_narrow`

Что сделать:
- найти позиции, для которых нет `order_id` в orders (left_anti)
- вывести: `order_id`, `order_item_id`, `product_id`

Куда записать:
- CSV (header=True): `/workspace/lesson03_03/task03_items_without_orders_csv`


In [ ]:
df_orders_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

df_items_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDER_ITEMS_CSV)
)

print(f"df_orders_narrow: {df_orders_narrow.count()}, df_items_narrow: {df_items_narrow.count()}")

df_join = df_items_narrow.join(df_orders_narrow, on="order_id", how="left_anti")\
    .select(
        "order_id",
        "order_item_id",
        "product_id"
    )

df_join.show(5, truncate=False)
df_join.count()

reset_dir(TASK03_OUT)

(
    df_join
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(TASK03_OUT)
)

df_join_back = read_csv_dir(TASK03_OUT)

print(f"df_join: {df_join.count()}, df_join_back: {df_join_back.count()}")

## Задание 4. LEFT JOIN orders + customers

Вход:
- `df_orders_narrow`
- `df_customers_narrow`

Что сделать:
- left join по `customer_id`
- вывести: `order_id`, `customer_id`, `customer_state`, `order_status`, `order_purchase_timestamp`

Куда записать:
- Parquet: `/workspace/lesson03_03/task04_orders_with_customers_left_parquet`


In [ ]:
df_orders_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

df_customers_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMERS_CSV)
)

print(f"df_orders_narrow: {df_orders_narrow.count()}, df_customers_narrow: {df_customers_narrow.count()}")

df_join = df_orders_narrow.join(df_customers_narrow, on="customer_id", how="left")\
    .select(
        "order_id",
        "customer_id",
        "customer_state",
        "order_status",
        "order_purchase_timestamp"
    )

df_join.show(5, truncate=False)
df_join.count()

reset_dir(TASK04_OUT)

(
    df_join
    .write
    .mode("overwrite")
    .parquet(TASK04_OUT)
)

df_join_back = read_parquet_dir(TASK04_OUT)

print(f"df_join: {df_join.count()}, df_join_back: {df_join_back.count()}")


## Задание 5. Аггрегация: сколько позиций в заказе

Вход:
- `df_order_items` (или `df_items_narrow`)

Что сделать:
- получить таблицу: `order_id`, `items_cnt`
- `items_cnt` = количество строк `order_items` в заказе
- сортировка: `items_cnt` по убыванию, затем `order_id` по возрастанию

Куда записать:
- Parquet: `/workspace/lesson03_03/task05_orders_items_cnt_parquet`


In [ ]:
df_order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDER_ITEMS_CSV)
)

print(f"df_order_items: {df_order_items.count()}")

df_select = (df_order_items
    .select(
        "order_id",
        "order_item_id"
    )\
    .groupBy("order_id")
    .agg(F.count("*").alias("items_cnt"))
    .orderBy(F.col("items_cnt").desc())
)

df_select.show(5, truncate=False)

reset_dir(TASK05_OUT)

(
    df_select
    .write
    .mode("overwrite")
    .parquet(TASK05_OUT)
)

df_back = read_parquet_dir(TASK05_OUT)

print(f"df_select: {df_select.count()}, df_join_back: {df_back.count()}")

## Задание 6. Цепочка JOIN: orders + items + products (узко)

Вход:
- `df_orders_narrow`, `df_items_narrow`, `df_products_narrow`

Что сделать:
- join items→orders (inner) по `order_id`
- затем join к products (left) по `product_id`
- вывести: `order_id`, `order_item_id`, `product_id`, `product_category_name`, `order_status`

Куда записать:
- Parquet: `/workspace/lesson03_03/task06_chain_orders_items_products_parquet`


In [ ]:
df_orders_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

df_items_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDER_ITEMS_CSV)
)

df_products_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PRODUCTS_CSV)
)

print(f"df_orders_narrow: {df_orders_narrow.count()}, df_items_narrow: {df_items_narrow.count()}, df_products_narrow: {df_products_narrow.count()}")

df_join_1 = df_items_narrow.join(df_orders_narrow, on="order_id", how="inner")

df_join_2 = df_join_1.join(df_products_narrow, on="product_id", how="left")\
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "product_category_name",
        "order_status"
    )

df_join_2.show(5, truncate=False)
df_join_2.count()

reset_dir(TASK06_OUT)

(
    df_join_2
    .write
    .mode("overwrite")
    .parquet(TASK06_OUT)
)

df_join_back = read_parquet_dir(TASK06_OUT)

print(f"df_join_2: {df_join_2.count()}, df_join_back: {df_join_back.count()}")

## Задание 7. Топ категорий по числу позиций (top-20)

Вход:
- результат из задания 6 (или соберите заново items+products)

Что сделать:
- сгруппировать по `product_category_name`
- посчитать `items_cnt`
- сортировка: `items_cnt` по убыванию
- вывести top-20

Куда записать:
- Parquet: `/workspace/lesson03_03/task07_top_categories_parquet`


In [ ]:
df_join = read_parquet_dir(TASK06_OUT)
df_join.show(5, truncate=False)

df_select = (df_join
        .select(
        "order_id",
        "order_item_id",
        "product_category_name"
    )\
    .groupBy("product_category_name")
    .agg(F.count("*").alias("items_cnt"))
    .orderBy(F.col("items_cnt").desc())
    .limit(20)
)

df_select.show(5, truncate=False)

reset_dir(TASK07_OUT)

(
    df_select
    .write
    .mode("overwrite")
    .parquet(TASK07_OUT)
)

df_join_back = read_parquet_dir(TASK07_OUT)

print(f"df_select: {df_select.count()}, df_join_back: {df_join_back.count()}")


## Задание 8. Диагностическая выборка (top-50 по свежести)

Вход:
- `df_orders_items` или `df_items_enriched`

Что сделать:
- выбрать 6–8 колонок для отладки
- отсортировать по `order_purchase_timestamp` по убыванию
- `limit(50)`

Куда записать:
- Parquet: `/workspace/lesson03_03/task08_debug_sample_parquet`


In [ ]:
df_items_narrow = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDER_ITEMS_CSV)
)

df_select = (df_orders_narrow
        .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date"
    ).orderBy(F.col("order_purchase_timestamp").desc()).limit(50)
)

df_select.show(5, truncate=False)

reset_dir(TASK08_OUT)

(
    df_select
    .write
    .mode("overwrite")
    .parquet(TASK08_OUT)
)

df_join_back = read_parquet_dir(TASK08_OUT)

print(f"df_select: {df_select.count()}, df_join_back: {df_join_back.count()}")

---
# ✅ Проверки

Запускайте check-ячейки ниже: получите **OK** или **НЕ OK** + причину.

Важно:
- проверки читают результаты из путей `TASK01_OUT ... TASK08_OUT`
- порядок строк после записи/чтения **не гарантирован**


In [ ]:
# Утилиты для проверки
def ok(task: str, details: str = "") -> None:
    msg = f"OK — {task}"
    if details:
        msg += f" | {details}"
    print(msg)

def bad(task: str, err: Exception) -> None:
    print(f"НЕ OK — {task} | {err}")

def run_check(task: str, check_fn) -> None:
    try:
        check_fn()
        ok(task)
    except AssertionError as e:
        bad(task, e)

def assert_exists(path: str) -> None:
    assert os.path.exists(path), f"путь не найден: {path}"


## Проверка задания 1 — orders + items (inner join) → Parquet

In [ ]:
def check_task01():
    assert_exists(TASK01_OUT)
    back = read_parquet_dir(TASK01_OUT)

    expected = df_items_narrow.join(df_orders_narrow.select("order_id"), on="order_id", how="inner")
    assert back.count() == expected.count(), "count() после inner join не совпал"

    expected_cols = ["order_id", "order_item_id", "product_id", "price", "customer_id", "order_status"]
    assert back.columns == expected_cols, f"ожидаются колонки: {expected_cols}"

run_check("TASK01", check_task01)


## Проверка задания 2 — items + products (left join) → Parquet

In [ ]:
def check_task02():
    assert_exists(TASK02_OUT)
    back = read_parquet_dir(TASK02_OUT)

    assert back.count() >= df_items_narrow.count(), "left join не должен уменьшать число строк"

    expected_cols = ["order_id", "order_item_id", "product_id", "product_category_name", "price"]
    assert back.columns == expected_cols, f"ожидаются колонки: {expected_cols}"

run_check("TASK02", check_task02)


## Проверка задания 3 — items without orders (left_anti) → CSV

In [ ]:
def check_task03():
    assert_exists(TASK03_OUT)
    back = read_csv_dir(TASK03_OUT)

    expected_cols = ["order_id", "order_item_id", "product_id"]
    assert back.columns == expected_cols, f"ожидаются колонки: {expected_cols}"

    expected = df_items_narrow.join(df_orders_narrow.select("order_id"), on="order_id", how="left_anti")
    assert back.count() == expected.count(), "count() left_anti не совпал с эталоном"

run_check("TASK03", check_task03)


## Проверка задания 4 — orders + customers (left join) → Parquet

In [ ]:
def check_task04():
    assert_exists(TASK04_OUT)
    back = read_parquet_dir(TASK04_OUT)

    expected_cols = ["order_id", "customer_id", "customer_state", "order_status", "order_purchase_timestamp"]
    assert back.columns == expected_cols, f"ожидаются колонки: {expected_cols}"

    assert back.count() == df_orders_narrow.count(), "left join orders->customers должен сохранять число заказов"

run_check("TASK04", check_task04)


## Проверка задания 5 — items_cnt по order_id → Parquet

In [ ]:
def check_task05():
    assert_exists(TASK05_OUT)
    back = read_parquet_dir(TASK05_OUT)

    expected_cols = ["order_id", "items_cnt"]
    assert back.columns == expected_cols, f"ожидаются колонки: {expected_cols}"

    total = back.select(F.sum("items_cnt").alias("s")).first()["s"]
    assert int(total) == df_order_items.count(), "сумма items_cnt должна равняться числу строк order_items"

run_check("TASK05", check_task05)


## Проверка задания 6 — цепочка joins → Parquet

In [ ]:
def check_task06():
    assert_exists(TASK06_OUT)
    back = read_parquet_dir(TASK06_OUT)

    expected_cols = ["order_id", "order_item_id", "product_id", "product_category_name", "order_status"]
    assert back.columns == expected_cols, f"ожидаются колонки: {expected_cols}"

    expected = (
        df_items_narrow
        .join(df_orders_narrow.select("order_id", "order_status"), on="order_id", how="inner")
        .join(df_products_narrow, on="product_id", how="left")
        .select("order_id", "order_item_id", "product_id", "product_category_name", "order_status")
    )

    assert back.count() == expected.count(), "count() цепочки join не совпал с эталоном"

run_check("TASK06", check_task06)


## Проверка задания 7 — top-20 categories → Parquet

In [ ]:
def check_task07():
    assert_exists(TASK07_OUT)
    back = read_parquet_dir(TASK07_OUT)

    expected_cols = ["product_category_name", "items_cnt"]
    assert back.columns == expected_cols, f"ожидаются колонки: {expected_cols}"

    assert back.count() <= 20, "должно быть не больше 20 строк (top-20)"
    assert back.filter(F.col("items_cnt").isNull()).count() == 0, "items_cnt не должен быть NULL"

run_check("TASK07", check_task07)


## Проверка задания 8 — debug sample top-50 → Parquet

In [ ]:
def check_task08():
    assert_exists(TASK08_OUT)
    back = read_parquet_dir(TASK08_OUT)

    assert back.count() <= 50, "должно быть не больше 50 строк"
    assert "order_id" in back.columns, "в результате должна быть колонка order_id"

run_check("TASK08", check_task08)
